# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 0. 제출 정보 (Chapter 04 제출 답안)

- 이름: 장태진
- GitHub ID: Taejin-AI
- 개인 저장소: `llm-data-analysis-study`
- 작성일: 2026.10.07
- 운영체제 / 실행 환경: Windows 10, 프로젝트 `.venv` 커널 (실제 경로와 버전은 아래 1번 셀과 마지막 Evidence 셀 Output 참고)
- 최종 제출 URL:

```text
https://github.com/Taejin-AI/llm-data-analysis-study/blob/master/chapter04/chapter04.ipynb
```

### 이 Notebook을 만든 방법

- 공식 `notebooks/ch04_pandas_basic.ipynb`를 복사하고, 강사 저장소 안(`notebooks/ch04/`)에서 실행했습니다. 그래서 `find_project_root()`가 `data/raw`, `reports`를 찾습니다. 실행이 끝난 뒤 이 파일을 개인 저장소 `chapter04/chapter04.ipynb`로 옮겼습니다.
- 공식 셀은 그대로 두었습니다. 바꾼 곳은 고객 정렬 결과에서 `name` 컬럼을 빼고 출력한 한 곳뿐입니다(개인정보 최소화).
- `[추가 검증]`, `[추가 분석]`, `[LLM 코드 검증]`, `과제 N` 표시가 붙은 셀은 답안 템플릿(`templates/chapter04_assignment.md`)과 실습 가이드 STEP 7·9·10·11·12·13의 성공 기준을 확인하려고 제가 추가한 셀입니다.
- 이번 장의 Evidence는 대부분 Notebook Output에 남아 있어서 템플릿의 이미지 칸은 해당 셀 Output으로 대신했습니다. LLM Prompt와 응답도 이 Notebook 안에 텍스트로 기록했습니다(6번 항목).

> 이 Notebook에서 `total_sales`는 **completed 주문에 포함된 `line_total`(quantity × unit_price)의 합계**입니다. 회계상 순매출이 아닙니다.

## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: /usr/bin/python3
현재 작업 폴더: /sessions/rcw-01jaueve2ey8aeytunn1muvw/mnt/dev/llm-data-analysis-course/notebooks/ch04
프로젝트 루트: /sessions/rcw-01jaueve2ey8aeytunn1muvw/mnt/dev/llm-data-analysis-course
데이터 폴더: /sessions/rcw-01jaueve2ey8aeytunn1muvw/mnt/dev/llm-data-analysis-course/data/raw
결과 저장 폴더: /sessions/rcw-01jaueve2ey8aeytunn1muvw/mnt/dev/llm-data-analysis-course/reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문
1. **completed 주문 기준**으로 어떤 카테고리·상품·월·고객에서 주문 금액이 많이 나왔는가?
2. 전체 주문 금액 중 취소·환불로 빠지는 비중은 얼마이고, 월별 금액 변화는 주문 수 때문인가, 완료율 때문인가?

### 사용한 파일/컬럼
| 파일 | shape | 사용한 컬럼 | 용도 |
| --- | --- | --- | --- |
| `order_items.csv` | (764, 5) | `order_id`, `product_id`, `quantity`, `unit_price` | 금액 계산(`line_total`)의 기준 행 |
| `orders.csv` | (300, 5) | `order_id`, `customer_id`, `order_date`, `order_status`, `payment_method` | 주문 상태 필터, 월, 고객 연결, 결제수단(과제 3) |
| `products.csv` | (100, 4) | `product_id`, `product_name`, `category`, `price` | 카테고리·상품명 연결 |
| `customers.csv` | (150, 6) | `customer_id`, `age`, `city` (+ 필터 실습용 `gender`) | 고객별 집계에 도시만 연결 |

### 결과 관찰
- 4개 파일 모두 기대 컬럼이 있어서 `KeyError` 없이 통과했습니다.
- 가이드 예시에 없던 컬럼이 실제로는 있었습니다: `customers.name`, `customers.signup_date`, `orders.payment_method`, `order_items.order_item_id`.
- `order_items`는 764행이고 `orders`는 300행입니다. 한 주문에 여러 상품 행이 붙는 구조(1:N)입니다.

### 나의 해석과 판단
- 금액은 상품 정가(`products.price`)가 아니라 실제 주문 단가(`order_items.unit_price`)로 계산해야 한다고 판단했습니다. 할인 등으로 두 값이 다를 수 있기 때문입니다. 이번 데이터에서는 두 값이 다른 행이 0건이었습니다(5번 추가 검증 셀).
- `customers.name`은 집계 질문에 필요하지 않아서 분석·출력·저장 어디에도 쓰지 않았습니다. 고객은 `customer_id` 기반 라벨로만 표시했습니다.
- 집계의 기준 행(grain)은 `order_items`의 1행(주문 상세)으로 정했습니다. 주문 단위 지표(`order_count`)는 `nunique`로 따로 계산해야 합니다.

### 업무·분석적 의미
- 같은 "매출"이라는 말도 어떤 테이블의 어떤 컬럼을 기준으로 하느냐에 따라 숫자가 달라집니다. 분석에 들어가기 전에 질문 → 기준 행 → 계산 컬럼을 먼저 고정해야 보고서마다 숫자가 달라지는 문제를 막을 수 있습니다.

### 한계와 추가 확인 사항
- 데이터는 샘플 생성 스크립트로 만든 합성 데이터입니다. 실제 업무 데이터에서 흔한 부분 환불, 할인, 배송비, 반품 일자 같은 컬럼이 없습니다.
- 주문이 상태 변경 이력 없이 최종 상태 하나만 갖고 있어서, 언제 취소·환불되었는지는 알 수 없습니다.

## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
# [제출자 수정] 고객 정렬 결과에서 불필요한 개인정보(name)는 출력하지 않습니다.
display(customers.drop(columns=['name']).sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,gender,age,city,signup_date
14,15,M,69,서울,2026-06-04
8,9,M,69,서울,2025-10-19
54,55,M,69,광주,2023-07-31
78,79,M,69,부산,2023-07-14
136,137,M,68,울산,2024-11-28
45,46,M,67,대전,2026-05-13
132,133,M,67,성남,2026-05-29
123,124,M,67,대구,2024-09-24
11,12,F,66,대전,2024-09-30
57,58,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건: `age >= 30` → 111명, `city == '서울'` → 15명, `city.isin(['서울','부산'])` → 31명 (필터 전에 `value_counts()`로 실제 표기가 `서울`, `부산` 한글인 것을 확인)
- 정렬 기준: `products.price` 내림차순 Top 10, `customers.age` 내림차순 Top 10
- 만든 파생 컬럼: `line_total`
- `line_total` 계산식: `quantity × unit_price` (예: order_id 1의 첫 행 3 × 102,000 = 306,000)
- Evidence: 위 4·5·6번 셀 Output

### 결과 관찰
- 도시는 10개이고 가장 많은 성남이 21명, 가장 적은 고양이 11명입니다. 서울은 15명으로 상위 도시가 아닙니다.
- 주문 상태는 `completed` 184, `cancelled` 64, `refunded` 52, 세 가지만 있습니다.
- 주문 상태와 무관한 전체 주문 상세 금액은 **255,610,000**입니다.
- 가격 상위 10개 상품은 모두 188,000~200,000으로 가격대가 몰려 있고, 뷰티와 스포츠가 7개를 차지합니다.

### 나의 해석과 판단
- `서울` 대신 `Seoul`이나 `서울시` 같은 다른 표기가 섞여 있었다면 필터 결과가 조용히 줄었을 것입니다. 오류 없이 0건 또는 일부만 나오기 때문에 더 위험합니다. 그래서 필터 값은 반드시 `value_counts()`/`unique()`로 확인한 뒤 쓰기로 했습니다.
- `age >= 30`과 `age > 30`처럼 경계값 하나만 달라도 인원이 바뀝니다. 연령 기준은 업무 정의(예: "30대 이상")와 맞는지 먼저 정해야 합니다.
- 255,610,000은 취소·환불이 포함된 금액이어서 "매출"이라고 부르면 안 된다고 판단했습니다. 실제로 completed 기준 금액은 이 값의 58.29%였습니다(아래 3·5번).

### 업무·분석적 의미
- 파생 컬럼을 언제 만들었는지(상태 연결 전/후)에 따라 같은 이름의 합계가 서로 다른 의미가 됩니다. 보고서에는 변수명보다 계산 범위를 먼저 적어야 합니다.

### 한계와 추가 확인 사항
- 이 데이터에는 결측과 이상 표기가 없었습니다. 실제 데이터라면 공백·대소문자·약칭이 섞인 도시명을 정규화하는 작업이 필요합니다(Chapter 05에서 다룸).
- `quantity`는 1~5, `unit_price`는 5,000 이상으로 음수나 0이 없었습니다. 반품을 음수 수량으로 기록하는 시스템이라면 `line_total` 해석이 달라집니다.

## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


In [9]:
# [추가 검증] completed 필터 범위와 주문 상태별 금액 구성 확인
print('completed_order_sales 주문 상태 값:', completed_order_sales['order_status'].unique().tolist())
print('order_month 결측:', completed_order_sales['order_month'].isna().sum())
print('완료 주문 수(고유 order_id):', completed_order_sales['order_id'].nunique())

status_amount = (
    order_sales
    .groupby('order_status', as_index=False)
    .agg(order_count=('order_id', 'nunique'), item_rows=('order_id', 'size'), amount=('line_total', 'sum'))
)
status_amount['amount_ratio(%)'] = (status_amount['amount'] / status_amount['amount'].sum() * 100).round(2)
display(status_amount)
print('order_date 범위:', order_sales['order_date'].min().date(), '~', order_sales['order_date'].max().date())

completed_order_sales 주문 상태 값: ['completed']
order_month 결측: 0
완료 주문 수(고유 order_id): 184


,order_status,order_count,item_rows,amount,amount_ratio(%)
0,cancelled,64,162,62181000,24.33
1,completed,184,474,148990000,58.29
2,refunded,52,128,44439000,17.39


order_date 범위: 2025-07-09 ~ 2026-07-08


## 9. 상품 데이터 병합과 검증


In [10]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 3. merge 검증

| 병합 | key | `validate` | 병합 전 → 후 행 수 | `_merge` 결과 | 판정 |
| --- | --- | --- | --- | --- | --- |
| `order_items` ← `orders` | `order_id` | `many_to_one` 통과 (`orders.order_id` 중복 0) | 764 → 764 | both 764 / left_only 0 | PASS |
| `completed_order_sales` ← `products` | `product_id` | `many_to_one` 통과 (`products.product_id` 중복 0) | 474 → 474 | both 474 / left_only 0 | PASS |
| `customer_sales` ← `customers[['customer_id','city']]` | `customer_id` | `one_to_one` 통과 | 100 → 100 | `city` 결측 0 | PASS |

- Evidence: 위 7·8·9번 셀과 `[추가 검증]` 셀 Output

### 결과 관찰
- 두 번의 many-to-one 병합 모두 행 수가 그대로였고, 미매칭(`left_only`)이 0건이었습니다.
- completed 필터 후에는 `order_status`에 `completed`만 남았습니다. `order_month` 결측과 날짜 변환 실패도 0건입니다.
- 주문 날짜 범위는 **2025-07-09 ~ 2026-07-08**입니다.

### 나의 해석과 판단
이번 병합이 안전하다고 판단한 근거는 네 가지입니다.
1. 오른쪽 테이블 key가 고유하다(중복 0). 그래서 왼쪽 행이 복제될 수 없다. `validate`가 이를 강제로 확인했다.
2. 병합 전후 행 수가 같다. 행이 복제되지도, 사라지지도 않았다.
3. `left_only` 0건. 모든 주문 상세가 주문·상품 정보를 받았다. `how='left'`인데 미매칭이 있었다면 `order_status`가 NaN이 되어 completed 필터에서 조용히 빠졌을 것이다.
4. 병합 후 금액 합계가 병합 전과 같다(148,990,000 → 148,990,000, 5번 교차 검증).

행 수가 늘거나 줄지 않았으므로 따로 설명할 증감은 없습니다.

### 업무·분석적 의미
- 오른쪽 key가 중복된 상태로 병합하면 행이 복제되어 금액이 부풀려집니다. 예를 들어 상품 마스터에 같은 `product_id`가 두 번 있으면 그 상품 주문 금액이 2배로 잡힙니다. 이 경우 오류 없이 실행되므로 `validate`가 없으면 발견하기 어렵습니다.
- 반대로 `how='inner'`로 미매칭을 조용히 버리면 금액이 줄어듭니다. 병합 방식은 "어떤 행을 잃어도 되는가"라는 업무 판단입니다.

### 한계와 추가 확인 사항
- `validate`는 key의 **개수 관계**만 확인합니다. key는 맞는데 다른 주문/상품이 연결된 경우(잘못된 ID 매핑)는 잡지 못합니다.
- `orders.customer_id`가 `customers`에 모두 존재하는지도 확인했습니다(`city` 결측 0). 다만 고객 마스터가 분석 시점 기준인지, 주문 시점 기준인지(이사로 도시가 바뀐 경우)는 확인할 수 없습니다.

## 10. 카테고리별·상품별 매출


In [11]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [12]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


In [13]:
# [추가 분석] 월별 전체 주문 수와 주문 상태 구성 — 월별 금액 변화가 '주문 수' 때문인지 '완료율' 때문인지 분리
monthly_status = pd.crosstab(
    order_sales.drop_duplicates('order_id')['order_date'].dt.to_period('M').astype(str),
    order_sales.drop_duplicates('order_id')['order_status'],
)
monthly_status['all_orders'] = monthly_status.sum(axis=1)
monthly_status['completion_rate(%)'] = (monthly_status['completed'] / monthly_status['all_orders'] * 100).round(1)
display(monthly_status)
print('전체 주문 수 합계:', monthly_status['all_orders'].sum(), '/', len(orders))
print('전체 평균 완료율(%):', round(len(orders[orders['order_status'] == 'completed']) / len(orders) * 100, 1))

order_status,cancelled,completed,refunded,all_orders,completion_rate(%)
order_date,,,,,
2025-07,3,8,4,15,53.3
2025-08,5,18,1,24,75.0
2025-09,6,13,4,23,56.5
2025-10,10,26,3,39,66.7
2025-11,4,12,7,23,52.2
2025-12,9,14,3,26,53.8
2026-01,3,22,4,29,75.9
2026-02,5,17,7,29,58.6
2026-03,6,14,1,21,66.7


전체 주문 수 합계: 300 / 300
전체 평균 완료율(%): 61.3


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [14]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 4. completed 주문 범위와 집계

- 분석 범위 정의: `order_status == 'completed'`인 주문 184건, 주문 상세 474행. 금액 = `line_total` 합계 **148,990,000**
- Evidence: 위 10·11·12번 셀과 `[추가 분석]` 셀 Output

| 집계 | 핵심 결과 |
| --- | --- |
| 카테고리별 | 1위 스포츠 31,743,000 (21.31%) → 전자기기 17.72% → 생활용품 16.05% → 뷰티 15.69% → 식품 11.12% → 도서 11.00% → 최하위 패션 10,587,000 (7.11%) |
| 상품별 | 1위 스포츠 상품 041: 5,705,000 (수량 35). completed 주문이 1건 이상인 상품은 98개 (100개 중 2개는 completed 판매 없음) |
| 월별 | 최고 2025-10: 25,766,000 / 26건. 최저 2026-07: 2,188,000 / 2건(7/8까지만 존재). 2026-06은 2,826,000 / 4건 |
| 고객별 | completed 주문이 있는 고객 100명(전체 150명). 1위 Customer 117(성남): 4,100,000 / 5건. 상위 10명이 합계의 약 23% |

### 결과 관찰
- 스포츠와 패션 사이에 약 3배(31.7M vs 10.6M) 차이가 있습니다. 스포츠는 상품 수도 가장 많고(19개, 전체의 19%), 패션은 11개입니다.
- 식품은 상품이 7개(7%)뿐인데 금액 비중은 11.12%입니다. 4번 과제에서 확인한 평균 가격도 137,143으로 가장 높습니다.
- 월별 주문 수는 2025-10이 26건으로 가장 많았고, 평균 주문 금액(AOV)은 약 57만~110만 사이에서 움직입니다.
- 2026-06은 전체 주문이 16건으로 적은 데다 완료율이 **25.0%**(cancelled 6, refunded 6)로 전체 평균 61.3%보다 크게 낮았습니다.

### 나의 해석과 판단
- 가장 중요하다고 본 결과는 **2026-06의 급감**입니다. `[추가 분석]` 셀로 나눠 보니 원인이 하나가 아니었습니다. 주문 수 자체가 적은 데다(16건, 월평균 약 23건) 완료율이 낮았습니다. completed 금액만 보면 "수요 감소"로 읽기 쉽지만, 실제로는 취소·환불 증가가 같이 작용했습니다.
- 2025-07(7/9부터)과 2026-07(7/8까지)은 **부분 월**이어서 다른 달과 같은 기준으로 비교하면 안 된다고 판단했습니다. 특히 2026-07의 AOV 1,094,000은 주문 2건으로 계산한 값이라 신뢰하기 어렵습니다.
- 카테고리 순위는 "많이 팔리는 카테고리"라기보다 **상품 구성(상품 수) 영향이 섞인 결과**로 봤습니다. 식품은 상품당 completed 금액이 약 237만으로 가장 높았고, 패션은 약 96만으로 가장 낮았습니다.

### 업무·분석적 의미
- 다음 질문은 "왜 6월 취소·환불이 늘었는가"입니다. 결제수단, 카테고리, 특정 상품 중 어디에 몰려 있는지 쪼개 보는 것이 우선입니다.
- 패션은 상품 수와 상품당 금액 둘 다 낮습니다. 품목 구성 문제인지, 가격·노출 문제인지는 다음 분석에서 나눠 봐야 합니다.
- 상위 10명 고객이 약 23%를 차지합니다. 고객 집중도는 높지 않은 편이라 특정 고객 이탈 리스크는 크지 않아 보입니다.

### 한계와 추가 확인 사항
- **`total_sales`를 회계상 순매출과 같다고 단정할 수 없는 이유**: ① 할인·쿠폰·포인트 차감 컬럼이 없어 `unit_price`가 최종 결제가인지 알 수 없습니다. ② 배송비·부가세 처리 기준이 없습니다. ③ `refunded`는 전액 환불로만 표시되어 부분 환불을 반영할 수 없습니다. ④ 주문일 기준 집계라 회계상 매출 인식 시점(배송 완료·구매 확정일)과 다를 수 있습니다. ⑤ 최근 주문의 상태가 앞으로 바뀔 수 있습니다.
- 월별 주문이 2~39건으로 표본이 작아서 월 간 차이가 우연한 변동일 수 있습니다.

In [15]:
# [추가 검증] 여러 집계 결과의 total_sales 합계 교차 검증
base_total = completed_sales_items['line_total'].sum()
checks = {
    '완료 주문 상세 line_total (orders merge 후)': completed_order_sales['line_total'].sum(),
    '완료 주문 상세 line_total (products merge 후)': base_total,
    'category_sales 합계': category_sales['total_sales'].sum(),
    'product_sales 합계': product_sales['total_sales'].sum(),
    'monthly_summary 합계': monthly_summary['total_sales'].sum(),
    'customer_sales 합계': customer_sales['total_sales'].sum(),
}
total_check = pd.DataFrame({'항목': list(checks.keys()), '합계': list(checks.values())})
total_check['기준 대비 차이'] = total_check['합계'] - base_total
total_check['결과'] = total_check['기준 대비 차이'].eq(0).map({True: 'PASS', False: 'FAIL'})
display(total_check)

# 주문 수(고유 order_id)와 수량도 같은 방식으로 교차 확인
print('monthly order_count 합계:', monthly_summary['order_count'].sum(),
      '/ 완료 주문 고유 order_id:', completed_order_sales['order_id'].nunique())
print('customer order_count 합계:', customer_sales['order_count'].sum())
print('category total_quantity 합계:', category_sales['total_quantity'].sum(),
      '/ 완료 주문 상세 quantity 합계:', completed_sales_items['quantity'].sum())
print('customer_sales city 결측:', customer_sales['city'].isna().sum(), '/ 구매 고객 수:', len(customer_sales))
print('unit_price != products.price 행 수:', (completed_sales_items['unit_price'] != completed_sales_items['price']).sum())
assert (total_check['결과'] == 'PASS').all(), '총합 불일치 발생'

,항목,합계,기준 대비 차이,결과
0,완료 주문 상세 line_total (orders merge 후),148990000,0,PASS
1,완료 주문 상세 line_total (products merge 후),148990000,0,PASS
2,category_sales 합계,148990000,0,PASS
3,product_sales 합계,148990000,0,PASS
4,monthly_summary 합계,148990000,0,PASS
5,customer_sales 합계,148990000,0,PASS


monthly order_count 합계: 184 / 완료 주문 고유 order_id: 184
customer order_count 합계: 184
category total_quantity 합계: 1442 / 완료 주문 상세 quantity 합계: 1442
customer_sales city 결측: 0 / 구매 고객 수: 100
unit_price != products.price 행 수: 0


## 5. 총합 일치 검증

| 항목 | 합계 | 결과 |
| --- | ---: | --- |
| 원본 completed `line_total` 합계 (products merge 후) | 148,990,000 | 기준 |
| completed `line_total` 합계 (orders merge 후, products merge 전) | 148,990,000 | PASS |
| 카테고리 합계 | 148,990,000 | PASS |
| 상품별 합계 | 148,990,000 | PASS |
| 월별 합계 | 148,990,000 | PASS |
| 고객별 합계 | 148,990,000 | PASS |
| 차이 여부 | 모든 항목 차이 0 | **PASS** |

- 같은 방식으로 수량(1,442 = 1,442)과 주문 수(월별 `order_count` 합 184 = completed 고유 `order_id` 184)도 일치했습니다.
- Evidence: 위 `[추가 검증] 총합 교차 검증` 셀 Output. 셀 안의 `assert`가 불일치 시 실행을 멈추도록 해 두었습니다.

### 나의 해석과 판단
- 집계는 데이터를 "나누어 다시 더하는" 작업이므로, 어느 기준으로 나누든 합계는 원본과 같아야 합니다. 합계가 다르면 집계 과정에서 **행이 복제되었거나(merge 중복), 빠졌거나(필터·NaN 키 제외)** 둘 중 하나입니다. 표만 보고는 이를 알 수 없어서 총합 검증이 필요합니다.
- 특히 `groupby`는 기본값(`dropna=True`)이 키가 NaN인 행을 **조용히 제외**합니다. 그래서 오류 없이 합계만 줄어들 수 있습니다.
- 불일치 시 확인 순서:
  1. 기준 합계가 맞는지 확인합니다. 필터 범위가 completed가 맞는지, 어느 단계의 DataFrame인지 봅니다.
  2. 병합 전후 행 수와 key 중복 수를 다시 확인합니다(복제 여부).
  3. `_merge`의 `left_only`와 병합 후 NaN 컬럼 수를 확인합니다(미매칭 여부).
  4. groupby 키(`category`, `order_month`, `customer_id`)의 NaN 수를 확인합니다(조용한 제외).
  5. 마지막으로 차이 금액과 같은 금액의 행을 찾아 원인 행을 특정합니다.

## 13. 결과 저장하기


In [16]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 246


ch04_product_sales.csv True 4310
ch04_monthly_sales.csv True 428
ch04_customer_sales.csv True 3286


In [17]:
# [추가 검증] 저장한 CSV 존재·크기·재읽기 확인 (STEP 11)
for path in sorted(REPORT_DIR.glob('ch04_*.csv')):
    print(path.name, path.exists(), path.stat().st_size)

reread_rows = []
for filename, df in outputs.items():
    check = pd.read_csv(REPORT_DIR / filename)
    reread_rows.append({
        'file': filename,
        'shape': check.shape,
        'columns_same': check.columns.tolist() == df.columns.tolist(),
        'rows_same': len(check) == len(df),
        'total_sales_same': (check['total_sales'].sum() == df['total_sales'].sum()) if 'total_sales' in df else None,
    })
display(pd.DataFrame(reread_rows))
# 저장 파일에 개인정보(name)가 없는지 확인
print('저장 파일 중 name 컬럼 포함:', [f for f, df in outputs.items() if 'name' in df.columns])

ch04_category_sales.csv True 246


ch04_customer_sales.csv True 3286
ch04_monthly_sales.csv True 428
ch04_product_sales.csv True 4310


,file,shape,columns_same,rows_same,total_sales_same
0,ch04_category_sales.csv,"(7, 4)",True,True,True
1,ch04_product_sales.csv,"(98, 5)",True,True,True
2,ch04_monthly_sales.csv,"(13, 4)",True,True,True
3,ch04_customer_sales.csv,"(100, 5)",True,True,True


저장 파일 중 name 컬럼 포함: []


### 결과 저장 검증 (STEP 11)
- 4개 CSV(`ch04_category_sales.csv` 7행, `ch04_product_sales.csv` 98행, `ch04_monthly_sales.csv` 13행, `ch04_customer_sales.csv` 100행)가 모두 존재하고, 파일 크기가 0보다 큽니다.
- 다시 읽었을 때 컬럼·행 수·`total_sales` 합계가 저장 전 DataFrame과 같았습니다. 저장 파일에는 `name` 컬럼이 없습니다.
- 판단: "저장 셀이 실행됐다"와 "올바른 파일이 남았다"는 다른 이야기입니다. 그래서 다시 읽어서 비교했습니다. `utf-8-sig`로 저장해 Excel에서 열어도 한글이 깨지지 않게 했습니다.
- 한계: 결과 CSV(`reports/`)는 강사 저장소의 실행 산출물이라 개인 저장소에는 올리지 않았습니다. 같은 데이터로 Notebook을 다시 실행하면 같은 파일이 만들어집니다.

## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


In [18]:
# [LLM 코드 검증 1] 위 예시 코드를 그대로 실행하면 어떻게 되는가?
try:
    order_items.groupby('category')['line_total'].sum()
except KeyError as e:
    print('KeyError 발생:', e)
print("order_items에 'category' 컬럼 존재:", 'category' in order_items.columns)
print("products에 'category' 컬럼 존재:", 'category' in products.columns)
# 설령 products를 붙여 실행되더라도 주문 상태 필터가 없으면 취소·환불까지 합산됩니다.
naive = order_items.merge(products[['product_id', 'category']], on='product_id').groupby('category')['line_total'].sum()
print('필터 없이 계산한 카테고리 합계:', naive.sum(), '/ completed 기준 합계:', category_sales['total_sales'].sum())

KeyError 발생: 'category'
order_items에 'category' 컬럼 존재: False
products에 'category' 컬럼 존재: True
필터 없이 계산한 카테고리 합계: 255610000 / completed 기준 합계: 148990000


## 6. LLM pandas 코드 검증 (STEP 12)

### 6-1. 공식 Notebook 예시 코드 검증 결과 (바로 위 셀)
- `order_items.groupby('category')`는 `category`가 `products`에만 있어서 **KeyError**가 났습니다.
- `products`를 붙여서 실행되게 고쳐도 주문 상태 필터가 없으면 합계가 **255,610,000**이 됩니다. completed 기준 148,990,000보다 **106,620,000(약 72%) 과대 계산**입니다. "실행되는 코드"와 "맞는 코드"는 다릅니다.

### 6-2. 실습 가이드 Prompt로 LLM에 요청
- 사용 LLM: Claude (Anthropic). Cowork 세션에서 아래 Prompt를 그대로 입력했습니다.

```text
orders와 order_items를 사용해
completed 주문 기준 월별 금액을 계산하려고 합니다.

다음을 반드시 포함해 주세요.
- line_total = quantity × unit_price
- orders.order_id 고유성 확인
- many-to-one merge + indicator
- 병합 전후 행 수 확인
- 미매칭 확인
- 날짜 변환 실패 확인
- completed만 필터링
- 월별 total_sales와 고유 order_count 계산
- 결과를 회계상 순매출이라고 단정하지 않기
```

### 6-3. LLM 제안 코드 (원문)

```python
import pandas as pd

orders = pd.read_csv("data/raw/orders.csv")
order_items = pd.read_csv("data/raw/order_items.csv")

# 1) 파생 컬럼
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

# 2) 키 고유성 확인
assert orders["order_id"].is_unique, "orders.order_id 중복"

# 3) many-to-one merge + indicator
merged = order_items.merge(
    orders[["order_id", "order_date", "order_status"]],
    on="order_id", how="left", validate="many_to_one", indicator=True,
)

# 4) 병합 전후 행 수 / 5) 미매칭
print("병합 전:", len(order_items), "병합 후:", len(merged))
print("미매칭(left_only):", (merged["_merge"] == "left_only").sum())

# 6) 날짜 변환 실패
merged["order_date"] = pd.to_datetime(merged["order_date"], errors="coerce")
print("날짜 변환 실패:", merged["order_date"].isna().sum())

# 7) completed 필터
completed = merged[merged["order_status"] == "completed"].copy()
completed["order_month"] = completed["order_date"].dt.to_period("M").astype(str)

# 8) 월별 total_sales / 고유 order_count
monthly = (
    completed.groupby("order_month", as_index=False)
    .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
)

# total_sales는 completed 주문의 line_total 합계이며, 할인·배송비·부분환불이 반영된 회계상 순매출이 아닙니다.
```

아래 셀에서 ① 원문을 그대로 실행해 보고(실패 지점인 데이터 로딩부), ② 경로만 고친 뒤 Notebook의 `monthly_summary`와 월별로 대조했습니다. 고친 버전에서는 Notebook 변수를 덮어쓰지 않도록 변수명 앞에 `llm_`을 붙였습니다.

In [19]:
# [LLM 코드 검증 2] LLM 제안 코드를 수정 없이 실행 (원문 그대로)
llm_code_asis = """
import pandas as pd

orders = pd.read_csv("data/raw/orders.csv")
order_items = pd.read_csv("data/raw/order_items.csv")
"""
try:
    exec(llm_code_asis, {})
except FileNotFoundError as e:
    print('FileNotFoundError:', e)
    print('현재 작업 폴더:', Path.cwd())

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/orders.csv'
현재 작업 폴더: /sessions/rcw-01jaueve2ey8aeytunn1muvw/mnt/dev/llm-data-analysis-course/notebooks/ch04


In [20]:
# [LLM 코드 검증 3] 경로만 DATA_DIR로 수정한 LLM 코드 실행 후 Notebook 결과와 비교
llm_orders = pd.read_csv(DATA_DIR / "orders.csv")          # 수정: 상대경로 → DATA_DIR
llm_items = pd.read_csv(DATA_DIR / "order_items.csv")       # 수정: 상대경로 → DATA_DIR

# 1) 파생 컬럼
llm_items["line_total"] = llm_items["quantity"] * llm_items["unit_price"]

# 2) 키 고유성 확인
assert llm_orders["order_id"].is_unique, "orders.order_id 중복"

# 3) many-to-one merge + indicator
llm_merged = llm_items.merge(
    llm_orders[["order_id", "order_date", "order_status"]],
    on="order_id", how="left", validate="many_to_one", indicator=True,
)

# 4) 병합 전후 행 수 / 5) 미매칭
print("병합 전:", len(llm_items), "병합 후:", len(llm_merged))
print("미매칭(left_only):", (llm_merged["_merge"] == "left_only").sum())

# 6) 날짜 변환 실패
llm_merged["order_date"] = pd.to_datetime(llm_merged["order_date"], errors="coerce")
print("날짜 변환 실패:", llm_merged["order_date"].isna().sum())

# 7) completed 필터
llm_completed = llm_merged[llm_merged["order_status"] == "completed"].copy()
llm_completed["order_month"] = llm_completed["order_date"].dt.to_period("M").astype(str)

# 8) 월별 total_sales / 고유 order_count
llm_monthly = (
    llm_completed.groupby("order_month", as_index=False)
    .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
)

# total_sales는 completed 주문의 line_total 합계이며, 할인·배송비·부분환불이 반영된 회계상 순매출이 아닙니다.

# ---- Notebook 결과(monthly_summary)와 대조 ----
compare = monthly_summary[['order_month', 'total_sales', 'order_count']].merge(
    llm_monthly, on='order_month', how='outer', suffixes=('_notebook', '_llm'), indicator=True,
)
compare['sales_diff'] = compare['total_sales_llm'] - compare['total_sales_notebook']
compare['count_diff'] = compare['order_count_llm'] - compare['order_count_notebook']
display(compare)
print('월 키 불일치:', (compare['_merge'] != 'both').sum())
print('월별 금액 차이 합:', compare['sales_diff'].abs().sum(), '/ 주문 수 차이 합:', compare['count_diff'].abs().sum())

병합 전: 764 병합 후: 764
미매칭(left_only): 0
날짜 변환 실패: 0


,order_month,total_sales_notebook,order_count_notebook,total_sales_llm,order_count_llm,_merge,sales_diff,count_diff
0,2025-07,5869000,8,5869000,8,both,0,0
1,2025-08,15621000,18,15621000,18,both,0,0
2,2025-09,10190000,13,10190000,13,both,0,0
3,2025-10,25766000,26,25766000,26,both,0,0
4,2025-11,8812000,12,8812000,12,both,0,0
5,2025-12,11501000,14,11501000,14,both,0,0
6,2026-01,17423000,22,17423000,22,both,0,0
7,2026-02,9749000,17,9749000,17,both,0,0
8,2026-03,13429000,14,13429000,14,both,0,0
9,2026-04,17553000,23,17553000,23,both,0,0


월 키 불일치: 0
월별 금액 차이 합: 0 / 주문 수 차이 합: 0


### 6-4. 검증 결과

- LLM Prompt 요약: completed 주문 기준 월별 금액 계산. 병합 검증 항목 9가지를 명시함
- 제안 코드 요약: `line_total` 생성 → `is_unique` assert → `many_to_one` + `indicator` 병합 → 행 수·미매칭·날짜 실패 출력 → completed 필터 → 월별 `sum`/`nunique`
- 실제 컬럼/범위와 맞지 않은 부분:
  1. **상대경로 `data/raw/...`**: Notebook 작업 폴더가 `notebooks/ch04`여서 `FileNotFoundError`가 났습니다. LLM은 코드가 어디서 실행되는지 모릅니다.
  2. 병합 전후 행 수와 미매칭을 `print`만 하고 멈추지 않습니다. 값이 틀려도 끝까지 실행되어 틀린 결과가 만들어질 수 있습니다.
  3. `order_status` 실제 값(`value_counts`)을 확인하지 않고 `'completed'` 문자열을 가정했습니다. 이번 데이터에서는 맞았지만 `Completed`, `완료`처럼 표기가 다르면 0건이 됩니다.
  4. 데이터 기간이 2025-07-09 ~ 2026-07-08이라 첫 달과 마지막 달이 부분 월이라는 점을 알 수 없었습니다. 그래서 월별 비교 시 주의 문구가 없습니다.
- 수정한 내용: 경로를 `DATA_DIR`로 바꿨습니다(변수명은 `llm_` 접두어로만 변경, 계산 로직은 원문 그대로).
- 대조 결과: 13개 월 모두 `total_sales`, `order_count` 차이 0, 월 키 불일치 0 → **계산 로직은 정확함**
- **최종 판단: 수정 후 사용**. 경로를 고치고, 2번 항목의 `print`를 `assert`로 바꾸고, 부분 월 주석을 추가하는 조건입니다.

### 나의 해석과 판단
- LLM 코드가 실행된다고 해서 분석이 맞다고 볼 수 없습니다. 이유는 세 가지입니다. ① LLM은 **실제 데이터 값과 실행 환경을 보지 않고** 일반적인 구조를 가정합니다(경로, 상태값 표기, 기간). ② 6-1처럼 실행은 되지만 **계산 범위가 틀린 코드**는 오류 없이 그럴듯한 숫자를 냅니다. ③ 이번처럼 Prompt에 검증 항목을 자세히 적으면 로직은 맞게 나오지만, 그 Prompt를 쓸 수 있었던 것은 내가 먼저 데이터 구조와 계산 기준을 알고 있었기 때문입니다.
- 그래서 LLM 결과는 **독립적으로 계산한 기준값(Notebook 결과)과 숫자로 대조**한 뒤에만 채택해야 한다고 판단했습니다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [21]:
# 과제 1. 40세 이상 고객 추출 (개인정보 최소화: name 제외)
customers_over_40 = customers.loc[customers['age'] >= 40, ['customer_id', 'gender', 'age', 'city']]
print('40세 이상 고객 수:', len(customers_over_40), f"({len(customers_over_40) / len(customers):.1%})")
print('age 범위:', customers['age'].min(), '~', customers['age'].max())
display(customers_over_40.sort_values('age', ascending=False).head())

40세 이상 고객 수: 78 (52.0%)
age 범위: 19 ~ 69


,customer_id,gender,age,city
8,9,M,69,서울
14,15,M,69,서울
54,55,M,69,광주
78,79,M,69,부산
136,137,M,68,울산


In [22]:
# 과제 2. 상품 가격이 낮은 순서대로 10개
display(products.sort_values(['price', 'product_id'], ascending=[True, True]).head(10))

,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


In [23]:
# 과제 3. 결제수단별 주문 수와 비율 (전체 주문 / completed 주문)
print('payment_method 결측:', orders['payment_method'].isna().sum())
payment_all = orders['payment_method'].value_counts().rename('order_count').to_frame()
payment_all['ratio(%)'] = (payment_all['order_count'] / payment_all['order_count'].sum() * 100).round(2)

completed_orders = orders[orders['order_status'] == 'completed']
payment_completed = completed_orders['payment_method'].value_counts().rename('completed_count').to_frame()
payment_completed['completed_ratio(%)'] = (payment_completed['completed_count'] / payment_completed['completed_count'].sum() * 100).round(2)

payment_summary = payment_all.join(payment_completed)
payment_summary['completion_rate(%)'] = (payment_summary['completed_count'] / payment_summary['order_count'] * 100).round(2)
display(payment_summary)
print('검증 - 전체 주문 수:', payment_summary['order_count'].sum(), '/', len(orders),
      '| completed 주문 수:', payment_summary['completed_count'].sum(), '/', len(completed_orders))

payment_method 결측: 0


,order_count,ratio(%),completed_count,completed_ratio(%),completion_rate(%)
payment_method,,,,,
kakao_pay,79,26.33,49,26.63,62.03
naver_pay,77,25.67,51,27.72,66.23
bank_transfer,74,24.67,45,24.46,60.81
card,70,23.33,39,21.20,55.71


검증 - 전체 주문 수: 300 / 300 | completed 주문 수: 184 / 184


In [24]:
# 과제 4. 카테고리별 평균 상품 가격 (상품 마스터 기준)
category_price = (
    products.groupby('category', as_index=False)
    .agg(product_count=('product_id', 'nunique'),
         avg_price=('price', 'mean'),
         median_price=('price', 'median'),
         min_price=('price', 'min'),
         max_price=('price', 'max'))
    .sort_values('avg_price', ascending=False)
)
category_price['avg_price'] = category_price['avg_price'].round(0)
display(category_price)
print('상품 수 합계:', category_price['product_count'].sum(), '/', len(products))

,category,product_count,avg_price,median_price,min_price,max_price
4,식품,7,137143.0,147000.0,67000,197000
1,뷰티,16,117688.0,134000.0,5000,200000
6,패션,11,115909.0,115000.0,28000,198000
3,스포츠,19,111579.0,103000.0,10000,196000
0,도서,14,106857.0,118500.0,25000,174000
5,전자기기,17,101588.0,111000.0,5000,165000
2,생활용품,16,96438.0,89000.0,11000,188000


상품 수 합계: 100 / 100


In [25]:
# 과제 5. 완료 주문과 전체 주문의 금액 차이
all_amount = order_sales['line_total'].sum()
completed_amount = completed_order_sales['line_total'].sum()
print('전체 주문 상세 금액:', all_amount)
print('완료 주문 금액:', completed_amount)
print('차이(전체 - 완료):', all_amount - completed_amount, f"({(all_amount - completed_amount) / all_amount:.2%})")
print('차이 = cancelled + refunded 금액:',
      order_sales.loc[order_sales['order_status'].isin(['cancelled', 'refunded']), 'line_total'].sum())
print('검증 - all_order_amount(병합 전 계산)와 동일:', all_amount == all_order_amount)

전체 주문 상세 금액: 255610000
완료 주문 금액: 148990000
차이(전체 - 완료): 106620000 (41.71%)
차이 = cancelled + refunded 금액: 106620000
검증 - all_order_amount(병합 전 계산)와 동일: True


### 과제 결과 정리 (공식 Notebook 15번 실습 과제)

**과제 1. 40세 이상 고객**: 78명(전체 150명의 52.0%). 연령 범위는 19~69세입니다. 출력할 때 `name`은 제외했습니다.

**과제 2. 가격 낮은 상품 10개**: 최저가는 5,000(전자기기 상품 006, 뷰티 상품 028)이고, 10번째는 32,000입니다. 같은 가격일 때 순서가 매번 같도록 `product_id`를 보조 정렬 키로 넣었습니다.

**과제 3. 결제수단별 주문 수와 비율**
- 관찰: 전체 300건은 kakao_pay 79(26.33%), naver_pay 77(25.67%), bank_transfer 74(24.67%), card 70(23.33%)으로 고르게 나뉩니다. 완료율은 naver_pay 66.23%로 가장 높고 card 55.71%로 가장 낮습니다.
- 해석: 약 10%p 차이가 있지만 결제수단별 표본이 70~79건이라 우연한 변동일 수 있습니다. 이 결과만으로 "카드 결제가 취소를 유발한다"고 판단하지 않았습니다.
- 다음 확인: 결제수단 × 월 × 상태를 교차해 2026-06 취소·환불 증가가 특정 결제수단에 몰렸는지 봅니다.

**과제 4. 카테고리별 평균 상품 가격** (상품 마스터 기준)
- 식품 137,143(7개) > 뷰티 117,688 > 패션 115,909 > 스포츠 111,579 > 도서 106,857 > 전자기기 101,588 > 생활용품 96,438
- 해석: 패션은 평균 가격이 3위인데 completed 금액은 최하위입니다. 가격보다 판매량(수량 111, 최하위)이 낮은 영향이 큽니다. 다만 이 값은 상품 정가 평균이고, 실제 판매 가중 평균이 아닙니다. 또 뷰티·전자기기는 최소 5,000에서 최대 20만까지 범위가 넓어 평균의 대표성이 낮습니다.

**과제 5. 완료 주문과 전체 주문의 금액 차이**
- 전체 255,610,000 − 완료 148,990,000 = **106,620,000 (41.71%)**. 이 값은 cancelled 62,181,000 + refunded 44,439,000과 정확히 일치합니다.
- 해석: 주문 건수 기준으로는 취소·환불이 38.7%(116/300)인데 금액 기준으로는 41.7%입니다. 취소·환불 주문이 평균적으로 조금 더 큰 주문이었을 가능성이 있습니다. 상태 필터 하나로 금액이 약 42% 달라지므로, 상태 기준을 명시하지 않은 "매출" 숫자는 보고에 쓸 수 없습니다.

**과제 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 Prompt**

```text
[데이터 구조]
- orders.csv: order_id(PK, 고유), customer_id, order_date, payment_method, order_status
  (order_status 실제 값: completed / cancelled / refunded)
- order_items.csv: order_item_id(PK), order_id(FK), product_id, quantity, unit_price
- customers.csv: customer_id(PK), name, gender, age, city, signup_date
- 파일 경로는 노트북에서 정의된 DATA_DIR(pathlib.Path)를 사용하세요. 상대경로를 쓰지 마세요.

[요청]
completed 주문 기준 고객별 구매 금액(total_sales)과 고유 주문 수(order_count)를 계산하는 pandas 코드를 작성해 주세요.

[반드시 포함]
1. line_total = quantity × unit_price
2. orders.order_id, customers.customer_id 중복 수 확인
3. order_items ← orders: how='left', validate='many_to_one', indicator=True
4. 병합 전후 행 수와 left_only 건수를 assert로 확인 (불일치 시 중단)
5. completed 필터 전 order_status.value_counts() 출력
6. 고객별 groupby 후 customers에서 customer_id, city만 validate='one_to_one'으로 연결
7. 고객별 total_sales 합계 == completed line_total 합계 assert
8. name 등 개인정보 컬럼은 결과에 포함하지 말고 customer_id 기반 라벨만 사용

[주의]
- total_sales를 회계상 순매출이라고 표현하지 마세요.
- 코드 실행 전 가정한 내용이 있으면 따로 목록으로 알려 주세요.

In [26]:
# [Chapter 04 Evidence] 직접 실행 결과 자동 요약 (STEP 13)
saved = {f: (REPORT_DIR / f).exists() for f in outputs}
evidence = f"""
[Chapter 04 Evidence]
1. Notebook
- 실행 Python: {sys.executable}
- pandas 버전: {pd.__version__}
2. 병합 검증
- orders.order_id 중복: {orders['order_id'].duplicated().sum()}
- orders merge 전/후 행 수: {len(order_items)} / {len(order_sales)}
- orders merge 미매칭: {len(order_items) - order_items['order_id'].isin(orders['order_id']).sum()}
- products merge 전/후 행 수: {len(completed_order_sales)} / {len(completed_sales_items)}
- products merge 미매칭: {len(completed_order_sales) - completed_order_sales['product_id'].isin(products['product_id']).sum()}
3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: {order_sales['order_date'].isna().sum()}
4. 교차 검증
- 완료 주문 상세 line_total 합계: {completed_sales_items['line_total'].sum():,}
- category_sales 합계: {category_sales['total_sales'].sum():,}
- monthly_summary 합계: {monthly_summary['total_sales'].sum():,}
- customer_sales 합계: {customer_sales['total_sales'].sum():,}
- 합계 일치 여부: {'PASS' if (total_check['결과'] == 'PASS').all() else 'FAIL'}
5. 저장 파일
""" + "\n".join(f"- {f}: {'존재' if ok else '없음'}" for f, ok in saved.items())
print(evidence)


[Chapter 04 Evidence]
1. Notebook
- 실행 Python: /usr/bin/python3
- pandas 버전: 2.3.3
2. 병합 검증
- orders.order_id 중복: 0
- orders merge 전/후 행 수: 764 / 764
- orders merge 미매칭: 0
- products merge 전/후 행 수: 474 / 474
- products merge 미매칭: 0
3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 0
4. 교차 검증
- 완료 주문 상세 line_total 합계: 148,990,000
- category_sales 합계: 148,990,000
- monthly_summary 합계: 148,990,000
- customer_sales 합계: 148,990,000
- 합계 일치 여부: PASS
5. 저장 파일
- ch04_category_sales.csv: 존재
- ch04_product_sales.csv: 존재
- ch04_monthly_sales.csv: 존재
- ch04_customer_sales.csv: 존재


## 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지
1. **상태 필터가 금액을 약 42% 바꾼다.** 전체 주문 상세 금액 255,610,000 중 completed는 148,990,000(58.29%)뿐이고, 106,620,000은 취소·환불입니다. LLM 예시 코드처럼 상태 필터를 빠뜨리면 오류 없이 금액이 약 72% 부풀려집니다.
2. **2026-06의 completed 금액 감소는 주문 수 감소와 완료율 급락이 겹친 결과다.** 2026-06은 주문 16건, 완료율 25.0%(전체 평균 61.3%)입니다. completed 금액만 보면 원인을 구분할 수 없어서, 상태별 월 분해가 필요했습니다.

### 그 결과를 뒷받침하는 수치/표
- `[추가 검증] completed 필터 범위` 셀의 상태별 금액표: completed 148,990,000 / cancelled 62,181,000 / refunded 44,439,000
- `[추가 분석] 월별 전체 주문 수와 주문 상태 구성` 표: 2026-06 cancelled 6, refunded 6, completed 4
- `[추가 검증] 총합 교차 검증` 표: 카테고리·상품·월·고객 합계 모두 148,990,000으로 일치(PASS). 위 해석의 숫자가 집계 오류에서 나온 것이 아님을 확인

### 추가로 확인하고 싶은 질문
- 2026-06의 취소·환불은 특정 카테고리·상품·결제수단에 몰려 있는가?
- 최근 주문(2026-06~07)의 상태는 아직 확정되지 않은 것인가? 확정까지 걸리는 기간은?
- 패션의 낮은 금액은 상품 수·판매량·가격 중 무엇 때문인가? (상품당 판매량, 판매 가중 평균 단가 비교)
- 일 단위로 바꿔 부분 월(2025-07, 2026-07)을 같은 일수 기준으로 비교하면 추세가 어떻게 보이는가?

### 현재 결과의 한계
- 합성 샘플 데이터(주문 300건)라 월별·결제수단별 표본이 작습니다. 차이가 우연한 변동일 수 있습니다.
- `total_sales`는 할인·배송비·부분 환불·매출 인식 시점이 반영되지 않은 **completed 주문 상세 금액**이고, 회계상 순매출이 아닙니다.
- 이번 분석은 집계와 비교까지만 했습니다. 원인(예: 6월 취소 증가의 이유)은 관찰된 패턴에 대한 **가설**일 뿐이고 검증되지 않았습니다.

## 최종 제출 체크
- [x] 핵심 셀 Output이 남아 있습니다. (오류 Cell 없음, LLM 검증의 예외는 `try/except`로 처리해 결과를 남김)
- [x] merge와 총합 검증 Evidence가 있습니다.
- [x] 결과 관찰과 해석이 구분되어 있습니다.
- [x] LLM 코드를 검증했습니다. (원문 실행 → 실패 원인 → 수정 → Notebook 결과와 월별 대조)
- [x] 개인정보/Secret이 없습니다. (`name` 미출력·미저장, API Key 없음)
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.
